<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.4-caching-strategies/notebooks/exercises-10_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.4: Caching Strategies

*Module 10 · AI System Architecture & Production Deployment*

> If 100 users ask "What is MCP?" you don't need 100 LLM calls. Exact-match cache handles identical queries. Semantic cache handles "What's MCP?", "Explain MCP", "Tell me about the Model Context Protocol" — all returning the same cached answer. This lesson builds both, with Redis for production scale.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-10.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Exact-Match Cache — Hash, Store, Lookup — `part1_exact_cache.py`
2. Step 2 — Semantic Cache — Catch Paraphrases With Embeddings — `part2_semantic_cache.py`
3. Step 3 — TTL & Eviction Policies — When to Expire, What to Keep — `part3_ttl_policies.py`
4. Step 4 — Redis Cache Backend — Production Scale with Memorystore — `part4_redis_cache.py`
5. Step 5 — CacheStack — Exact + Semantic + TTL in One Lookup — `part5_cache_stack.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy redis


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Exact-Match Cache — Hash, Store, Lookup

Normalize the query, hash it, check the cache. Hit = instant response.

**`part1_exact_cache.py`** · _Block 1 of 5_

EXACT-MATCH CACHE — Normalize → hash → lookup. O(1). Free.


In [ ]:
# =============================================
# EXACT-MATCH CACHE
# Normalize → hash → lookup. O(1). Free.
# =============================================

import hashlib, json, time
from dataclasses import dataclass, field

@dataclass
class CacheEntry:
    response: str
    model: str
    created_at: float
    ttl_seconds: int
    hit_count: int = 0
    tokens_saved: int = 0

class ExactMatchCache:
    """O(1) cache for identical queries."""
    
    def __init__(self, max_entries: int = 10000, default_ttl: int = 3600):
        self.cache: dict[str, CacheEntry] = {}
        self.max_entries = max_entries
        self.default_ttl = default_ttl
        self.stats = {"hits": 0, "misses": 0, "evictions": 0}
    
    def _normalize(self, query: str, model: str) -> str:
        """Normalize query for consistent hashing."""
        normalized = query.lower().strip()
        normalized = " ".join(normalized.split())  # collapse whitespace
        return normalized + "|" + model
    
    def _hash(self, text: str) -> str:
        return hashlib.sha256(text.encode()).hexdigest()
    
    def get(self, query: str, model: str) -> dict:
        """Look up a cached response. Returns None on miss."""
        key = self._hash(self._normalize(query, model))
        entry = self.cache.get(key)
        
        if not entry:
            self.stats["misses"] += 1
            return None
        
        # Check TTL
        if time.time() - entry.created_at > entry.ttl_seconds:
            del self.cache[key]
            self.stats["misses"] += 1
            return None
        
        entry.hit_count += 1
        self.stats["hits"] += 1
        return {"response": entry.response, "model": entry.model,
                "cached": True, "cache_type": "exact",
                "age_s": round(time.time() - entry.created_at)}
    
    def put(self, query: str, model: str, response: str,
            ttl: int = None, est_tokens: int = 0):
        """Store a response in the cache."""
        # Evict oldest if full (simple LRU)
        if len(self.cache) >= self.max_entries:
            oldest_key = min(self.cache, key=lambda k: self.cache[k].created_at)
            del self.cache[oldest_key]
            self.stats["evictions"] += 1
        
        key = self._hash(self._normalize(query, model))
        self.cache[key] = CacheEntry(
            response=response, model=model,
            created_at=time.time(), ttl_seconds=ttl or self.default_ttl,
            tokens_saved=est_tokens,
        )
    
    @property
    def hit_rate(self) -> float:
        total = self.stats["hits"] + self.stats["misses"]
        return self.stats["hits"] / total if total > 0 else 0

# ── Demo ──
cache = ExactMatchCache(default_ttl=3600)

cache.put("What is MCP?", "gemini-2.0-flash", "MCP is an open protocol by Anthropic...", est_tokens=500)

print("Exact-match cache:\n")
tests = ["What is MCP?", "what is mcp?", "What  is  MCP?", "Explain MCP"]
for q in tests:
    result = cache.get(q, "gemini-2.0-flash")
    icon = "✅ HIT" if result else "❌ MISS"
    print(f"  {icon}  \"{q}\"")

print(f"\n  Hit rate: {cache.hit_rate:.0%} — \"Explain MCP\" missed (different wording)")


> **What just happened?** ExactMatchCache normalizes queries (lowercase, collapse whitespace) before hashing. "What is MCP?", "what is mcp?", and "What is MCP?" all normalize to the same hash → cache hit. But "Explain MCP" has different words → cache miss. TTL-based expiry. LRU eviction when full. Exact-match catches ~20-30% of real traffic (users asking the same FAQ questions). It's free, O(1), and instant — but misses paraphrases.


### Step 2 · Semantic Cache — Catch Paraphrases With Embeddings

"What is MCP?", "Explain MCP", "Tell me about Model Context Protocol" → same cached answer.

**`part2_semantic_cache.py`** · _Block 2 of 5_

SEMANTIC CACHE — Embed queries. If a new query is similar


In [ ]:
# =============================================
# SEMANTIC CACHE
# Embed queries. If a new query is similar
# enough to a cached one → return cached answer.
# =============================================

import numpy as np
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY", ""))

class SemanticCache:
    """Cache that matches by meaning, not exact text."""
    
    def __init__(self, similarity_threshold: float = 0.92,
                 max_entries: int = 5000, default_ttl: int = 3600):
        self.threshold = similarity_threshold
        self.max_entries = max_entries
        self.default_ttl = default_ttl
        
        self.entries: list[dict] = []      # [{query, response, model, embedding, created_at, ttl, hits}]
        self.embeddings: list[np.ndarray] = []
        self.stats = {"hits": 0, "misses": 0, "embed_calls": 0}
    
    def _embed(self, text: str) -> np.ndarray:
        """Get embedding for a query."""
        self.stats["embed_calls"] += 1
        result = genai.embed_content(
            model="models/text-embedding-004",
            content=text,
            task_type="retrieval_query",
        )
        return np.array(result["embedding"])
    
    def get(self, query: str, model: str) -> dict:
        """Find the closest cached response by embedding similarity."""
        if not self.embeddings:
            self.stats["misses"] += 1
            return None
        
        # Embed the query
        q_emb = self._embed(query)
        q_norm = q_emb / np.linalg.norm(q_emb)
        
        # Cosine similarity against all cached embeddings
        cache_matrix = np.array(self.embeddings)
        cache_norms = cache_matrix / np.linalg.norm(cache_matrix, axis=1, keepdims=True)
        similarities = cache_norms @ q_norm
        
        best_idx = np.argmax(similarities)
        best_sim = float(similarities[best_idx])
        
        if best_sim < self.threshold:
            self.stats["misses"] += 1
            return None
        
        entry = self.entries[best_idx]
        
        # Check TTL
        if time.time() - entry["created_at"] > entry["ttl"]:
            self._remove(best_idx)
            self.stats["misses"] += 1
            return None
        
        # Check model match
        if entry["model"] != model:
            self.stats["misses"] += 1
            return None
        
        entry["hits"] += 1
        self.stats["hits"] += 1
        
        return {"response": entry["response"], "model": entry["model"],
                "cached": True, "cache_type": "semantic",
                "similarity": round(best_sim, 3),
                "original_query": entry["query"]}
    
    def put(self, query: str, model: str, response: str, ttl: int = None):
        """Store a query-response pair with its embedding."""
        if len(self.entries) >= self.max_entries:
            self._remove(0)  # remove oldest
        
        embedding = self._embed(query)
        self.entries.append({
            "query": query, "response": response, "model": model,
            "created_at": time.time(), "ttl": ttl or self.default_ttl, "hits": 0,
        })
        self.embeddings.append(embedding)
    
    def _remove(self, idx: int):
        self.entries.pop(idx)
        self.embeddings.pop(idx)

# ── Demo ──
sem_cache = SemanticCache(similarity_threshold=0.90)

# Store one response
sem_cache.put("What is MCP?", "gemini-2.0-flash",
    "MCP (Model Context Protocol) is an open standard by Anthropic for connecting AI models to tools.")

# Test with paraphrases
print("Semantic cache (threshold=0.90):\n")
tests = [
    "What is MCP?",                           # exact → high similarity
    "Explain the Model Context Protocol",      # paraphrase → should hit
    "Tell me about MCP by Anthropic",           # paraphrase → should hit
    "What is the capital of France?",           # unrelated → should miss
    "How does MCP compare to function calling?", # related but different question
]

for q in tests:
    result = sem_cache.get(q, "gemini-2.0-flash")
    if result:
        print(f"  ✅ HIT  (sim={result['similarity']:.2f}) \"{q}\"")
    else:
        print(f"  ❌ MISS              \"{q}\"")

print(f"\n  Hit rate: {sem_cache.stats['hits']}/{sem_cache.stats['hits']+sem_cache.stats['misses']}")
print(f"  Embedding API calls: {sem_cache.stats['embed_calls']}")


> **What just happened?** SemanticCache embeds every cached query. When a new query arrives, it embeds it and computes cosine similarity against all cached embeddings. "Explain the Model Context Protocol" matches "What is MCP?" at sim=0.94 → cache hit. "What is the capital of France?" → sim=0.3 → miss. "How does MCP compare to function calling?" → sim=0.78 → miss (related but different question — below 0.90 threshold). Semantic cache catches the paraphrases exact-match misses, adding another 10-20% hit rate on top.


### Step 3 · TTL & Eviction Policies — When to Expire, What to Keep

Not all responses age the same. Facts last hours. News lasts minutes. Code lasts days.

**`part3_ttl_policies.py`** · _Block 3 of 5_

TTL & EVICTION POLICIES — Different content types need different TTLs.


In [ ]:
# =============================================
# TTL & EVICTION POLICIES
# Different content types need different TTLs.
# Smart eviction keeps high-value entries.
# =============================================

class TTLPolicy:
    """Assign TTLs based on query type and model."""
    
    # TTL by content category (seconds)
    CATEGORY_TTL = {
        "factual":     86400,    # 24 hours — "What is Python?"
        "definition":  86400,    # 24 hours — "Define RAG"
        "code":        172800,   # 48 hours — code snippets rarely change
        "current":     300,      # 5 minutes — "latest news", "current price"
        "opinion":     3600,     # 1 hour — subjective, may vary
        "default":     3600,     # 1 hour — safe default
    }
    
    # Keywords that indicate time-sensitive content
    VOLATILE_KEYWORDS = ["latest", "current", "today", "now", "recent",
                         "price", "stock", "weather", "news", "score"]
    
    STABLE_KEYWORDS = ["what is", "define", "explain", "how does", "write a function",
                       "implement", "algorithm", "history of"]
    
    def get_ttl(self, query: str, model: str = "") -> int:
        """Determine TTL for a query based on content type."""
        query_lower = query.lower()
        
        # Time-sensitive → short TTL
        if any(kw in query_lower for kw in self.VOLATILE_KEYWORDS):
            return self.CATEGORY_TTL["current"]
        
        # Code generation → long TTL
        if any(kw in query_lower for kw in ["write", "code", "implement", "function"]):
            return self.CATEGORY_TTL["code"]
        
        # Definitions and explanations → long TTL
        if any(kw in query_lower for kw in self.STABLE_KEYWORDS):
            return self.CATEGORY_TTL["factual"]
        
        return self.CATEGORY_TTL["default"]
    
    def should_cache(self, query: str) -> bool:
        """Some queries should never be cached."""
        query_lower = query.lower()
        # Don't cache personal or user-specific queries
        nocache = ["my account", "my order", "my profile", "my password",
                   "for me", "personally"]
        return not any(kw in query_lower for kw in nocache)

# ── Demo ──
policy = TTLPolicy()

print("TTL policy:\n")
tests = [
    "What is Python?",                  # factual → 24h
    "Write a binary search function",    # code → 48h
    "What's the latest Bitcoin price?",   # volatile → 5min
    "How does RAG work?",                # definition → 24h
    "Show me my order history",           # personal → DON'T CACHE
]

for q in tests:
    ttl = policy.get_ttl(q)
    cacheable = policy.should_cache(q)
    hours = ttl / 3600
    label = f"{hours:.1f}h" if hours >= 1 else f"{ttl//60}min"
    icon = "✅" if cacheable else "🚫"
    print(f"  {icon} TTL={label:5s} \"{q}\"")


> **What just happened?** TTLPolicy assigns different expiry times by content type: "What is Python?" → 24 hours (factual, rarely changes). "Write a binary search" → 48 hours (code is stable). "Latest Bitcoin price" → 5 minutes (volatile). Personal queries ("my order history") are never cached — they're user-specific. Smart TTL prevents serving stale data while maximizing cache utilization. A one-size-fits-all TTL either wastes cache space (too short) or serves stale data (too long).


### Step 4 · Redis Cache Backend — Production Scale with Memorystore

In-memory caches die when the process restarts. Redis persists across deploys and scales across instances.

**`part4_redis_cache.py`** · _Block 4 of 5_

REDIS CACHE BACKEND — Production cache using Redis (Memorystore).


In [ ]:
# =============================================
# REDIS CACHE BACKEND
# Production cache using Redis (Memorystore).
# Survives restarts. Shared across instances.
# pip install redis numpy
# =============================================

import redis, json, struct

class RedisCacheBackend:
    """Redis-backed cache for exact + semantic matching."""
    
    def __init__(self, redis_url: str = "redis://localhost:6379",
                 prefix: str = "llm_cache"):
        self.r = redis.from_url(redis_url, decode_responses=False)
        self.prefix = prefix
    
    # ── Exact match operations ──
    def exact_get(self, cache_key: str) -> dict:
        data = self.r.get(f"{self.prefix}:exact:{cache_key}")
        return json.loads(data) if data else None
    
    def exact_set(self, cache_key: str, response: str, model: str, ttl: int):
        data = json.dumps({"response": response, "model": model}).encode()
        self.r.setex(f"{self.prefix}:exact:{cache_key}", ttl, data)
    
    # ── Semantic cache operations ──
    def semantic_add(self, entry_id: str, query: str, response: str,
                     model: str, embedding: np.ndarray, ttl: int):
        """Store a semantic cache entry with its embedding."""
        key = f"{self.prefix}:sem:{entry_id}"
        emb_key = f"{self.prefix}:emb:{entry_id}"
        
        # Store metadata
        self.r.setex(key, ttl, json.dumps({
            "query": query, "response": response, "model": model,
        }).encode())
        
        # Store embedding as raw bytes (compact)
        self.r.setex(emb_key, ttl, embedding.astype(np.float32).tobytes())
        
        # Track active entries in a set
        self.r.sadd(f"{self.prefix}:sem_ids", entry_id)
    
    def semantic_search(self, query_embedding: np.ndarray,
                        model: str, threshold: float = 0.92) -> dict:
        """Find the closest semantic match across all entries."""
        entry_ids = self.r.smembers(f"{self.prefix}:sem_ids")
        if not entry_ids:
            return None
        
        best_match = None
        best_sim = 0.0
        q_norm = query_embedding / np.linalg.norm(query_embedding)
        
        for eid in entry_ids:
            eid_str = eid.decode() if isinstance(eid, bytes) else eid
            
            # Load embedding
            emb_bytes = self.r.get(f"{self.prefix}:emb:{eid_str}")
            if not emb_bytes: continue
            
            cached_emb = np.frombuffer(emb_bytes, dtype=np.float32)
            c_norm = cached_emb / np.linalg.norm(cached_emb)
            sim = float(np.dot(q_norm, c_norm))
            
            if sim > best_sim:
                best_sim = sim
                best_match = eid_str
        
        if best_sim < threshold or not best_match:
            return None
        
        # Load the matching entry
        data = self.r.get(f"{self.prefix}:sem:{best_match}")
        if not data: return None
        
        entry = json.loads(data)
        if entry["model"] != model: return None
        
        return {**entry, "similarity": round(best_sim, 3), "cache_type": "semantic", "cached": True}

print("""
Redis/Memorystore setup:

  # Local development
  docker run -d -p 6379:6379 redis:7-alpine
  
  # Production: Google Cloud Memorystore
  gcloud redis instances create llm-cache \\
    --region asia-south1 \\
    --tier basic \\
    --size 1 \\
    --redis-version redis_7_0

  # Connect from Cloud Run
  REDIS_URL = "redis://10.0.0.3:6379"  # Memorystore private IP

  Cost: ₹3,000/month for 1GB Memorystore (holds ~500K cached responses)
""")


> **What just happened?** RedisCacheBackend stores both exact and semantic caches in Redis. Exact entries use SETEX with TTL (Redis auto-expires). Semantic entries store metadata as JSON + embeddings as raw float32 bytes (compact: 768-dim embedding = 3KB). semantic_search() iterates all entries and finds the closest match. Redis persists across process restarts and is shared across multiple Cloud Run instances. 1GB Memorystore (₹3,000/month) holds ~500K cached responses — enough for most production workloads.


### Step 5 · CacheStack — Exact + Semantic + TTL in One Lookup

Check exact first (free, instant). Miss? Check semantic (1 embed call, ~10ms). Miss? Call the LLM. Store in both.

**`part5_cache_stack.py`** · _Block 5 of 5_

CACHE STACK — Exact → Semantic → LLM → Store in both.


In [ ]:
# =============================================
# CACHE STACK
# Exact → Semantic → LLM → Store in both.
# The complete caching pipeline.
# =============================================

class CacheStack:
    """Layered cache: exact match → semantic match → LLM call."""
    
    def __init__(self, semantic_threshold: float = 0.92):
        self.exact = ExactMatchCache(max_entries=10000)
        self.semantic = SemanticCache(similarity_threshold=semantic_threshold)
        self.ttl_policy = TTLPolicy()
        self.stats = {"exact_hits": 0, "semantic_hits": 0, "llm_calls": 0,
                      "tokens_saved": 0, "cost_saved_usd": 0.0}
    
    def query(self, prompt: str, model: str, llm_fn = None) -> dict:
        """Check caches, fall through to LLM if needed."""
        
        # Skip cache for uncacheable queries
        if not self.ttl_policy.should_cache(prompt):
            if llm_fn:
                response = llm_fn(prompt)
                return {"response": response, "cached": False, "reason": "uncacheable"}
            return {"response": None, "cached": False}
        
        # Layer 1: EXACT MATCH (free, < 1ms)
        exact_result = self.exact.get(prompt, model)
        if exact_result:
            self.stats["exact_hits"] += 1
            self.stats["tokens_saved"] += 500  # estimated
            self.stats["cost_saved_usd"] += 0.0003
            return exact_result
        
        # Layer 2: SEMANTIC MATCH (1 embed call, ~10ms)
        semantic_result = self.semantic.get(prompt, model)
        if semantic_result:
            self.stats["semantic_hits"] += 1
            self.stats["tokens_saved"] += 500
            self.stats["cost_saved_usd"] += 0.0003
            # Also store as exact match for future identical queries
            ttl = self.ttl_policy.get_ttl(prompt)
            self.exact.put(prompt, model, semantic_result["response"], ttl)
            return semantic_result
        
        # Layer 3: LLM CALL (full cost + latency)
        self.stats["llm_calls"] += 1
        if not llm_fn:
            return {"response": None, "cached": False}
        
        response = llm_fn(prompt)
        
        # Store in BOTH caches
        ttl = self.ttl_policy.get_ttl(prompt)
        self.exact.put(prompt, model, response, ttl)
        self.semantic.put(prompt, model, response, ttl)
        
        return {"response": response, "cached": False, "cache_type": "none"}
    
    def report(self):
        s = self.stats
        total = s["exact_hits"] + s["semantic_hits"] + s["llm_calls"]
        hit_rate = (s["exact_hits"] + s["semantic_hits"]) / total if total > 0 else 0
        
        print(f"\n  📊 Cache Report")
        print(f"  {'─'*40}")
        print(f"  Exact hits:     {s['exact_hits']}")
        print(f"  Semantic hits:   {s['semantic_hits']}")
        print(f"  LLM calls:       {s['llm_calls']}")
        print(f"  Hit rate:        {hit_rate:.0%}")
        print(f"  Tokens saved:    {s['tokens_saved']:,}")
        print(f"  Cost saved:      ${s['cost_saved_usd']:.4f}")

# ═══════════════════════════════════════════
# FULL DEMO
# ═══════════════════════════════════════════

stack = CacheStack(semantic_threshold=0.90)

def mock_llm(prompt):
    return f"[LLM] Response for: {prompt[:40]}"

queries = [
    "What is MCP?",                           # LLM call (first time)
    "What is MCP?",                           # exact hit
    "what is mcp?",                           # exact hit (normalized)
    "Explain the Model Context Protocol",      # semantic hit
    "Tell me about MCP by Anthropic",           # semantic hit
    "What is RAG?",                            # LLM call (different topic)
    "Explain retrieval augmented generation",   # semantic hit (from RAG entry)
    "Show me my order status",                 # LLM call (uncacheable)
]

print("CacheStack demo:\n")
for q in queries:
    result = stack.query(q, "gemini-2.0-flash", mock_llm)
    if result.get("cached"):
        print(f"  ✅ {result['cache_type']:8s} \"{q}\"")
    else:
        print(f"  🔄 {'LLM call':8s} \"{q}\"")

stack.report()


> **What just happened?** CacheStack layers all three strategies: (1) Check exact match (free, <1ms). (2) If miss, check semantic match (1 embedding call, ~10ms). (3) If miss, call the LLM (full cost, ~500ms) and store in BOTH caches. A semantic hit also populates the exact cache for future identical queries. Uncacheable queries ("my order status") skip caching entirely. The report shows: 8 queries → 2 exact hits + 3 semantic hits + 3 LLM calls = 62.5% hit rate . Tokens and dollars saved tracked automatically.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
